### Ejemplo básico

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

#crea un experimento, no tiene que existir
#mlflow.create_experiment("mi_experimento")
#tambiénm crea un experimento, puede existir
mlflow.set_experiment("mi_experimento")
mlflow.set_experiment_tag("scikit-learn","lr")
# Tags
mlflow.set_tag("tipo", "basico")
#ejecucion
mlflow.end_run()
with mlflow.start_run(run_name="run_primera"):
    mlflow.log_param("depth", 6)
    mlflow.log_metric("auc", 0.81)


2026/02/05 18:05:48 INFO mlflow.tracking.fluent: Experiment with name 'mi_experimento1' does not exist. Creating a new experiment.


🏃 View run sassy-ray-183 at: http://127.0.0.1:5000/#/experiments/2/runs/c2f2fbcc08be4bafa7dc088015672d2f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run run_primera at: http://127.0.0.1:5000/#/experiments/2/runs/c09b7177c8ea494d92de10217258746f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [4]:
#Crea una carpeta en disco ./mlruns
mlflow.set_tracking_uri("file:./mlruns")  # crea/usa ./mlruns en tu proyecto


In [5]:
import mlflow

print("Tracking URI:", mlflow.get_tracking_uri())
df = mlflow.search_runs(max_results=20)
print(df[["run_id", "status", "metrics.accuracy"]].head())

Tracking URI: file:./mlruns


c:\Users\tteru\.pyenv\pyenv-win\versions\3.12.8\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)


KeyError: "['metrics.accuracy'] not in index"

### Randomforest 

In [6]:
import os
import json
from pathlib import Path
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

import matplotlib.pyplot as plt

In [7]:
def main():
    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    mlflow.set_experiment("iris RandomForestClassifier")

    X, y = load_iris(return_X_y=True)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

    with mlflow.start_run(run_name="run_random"):
        params = {"n_estimators": 200, "max_depth": 5, "random_state": 42}
        mlflow.log_params(params)
    
        
        model = RandomForestClassifier(**params)
        model.fit(Xtr, ytr)
        #acc
        acc = model.score(Xte, yte)
        print(f'acc: {acc}')
        #crea una metrica accuracy
        mlflow.log_metric("accuracy", acc)
        
        # Tags
        mlflow.set_tag("dataset", "load_iris")
        #predict
        y_pred = model.predict(Xte)
        # Artefactos: matriz de confusión (png)
        cm = confusion_matrix(yte, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot()
        Path("artifacts").mkdir(exist_ok=True)
        fig_path = Path("artifacts") / "confusion_matrix_.png"
        plt.savefig(fig_path, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(str(fig_path), artifact_path="figures")
    
        #guarda el modelo en mlflow
        mlflow.sklearn.log_model(model, artifact_path="model")
        #cargo el modelo
        print(model)


In [8]:
if __name__ == "__main__":    
    main()

2026/02/05 18:06:59 INFO mlflow.tracking.fluent: Experiment with name 'iris RandomForestClassifier' does not exist. Creating a new experiment.


acc: 1.0


2026/02/05 18:07:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\tteru\.pyenv\pyenv-win\versions\3.12.8\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


RandomForestClassifier(max_depth=5, n_estimators=200, random_state=42)
🏃 View run run_random at: http://127.0.0.1:5000/#/experiments/3/runs/3506ccfbd543451a9199d8c25e33dd88
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


### Regresión

In [9]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("iris LogisticRegression")

X, y  = load_iris(return_X_y=True)
model = LogisticRegression(max_iter=200).fit(X, y)

#guarda metricas y parametros 
mlflow.sklearn.autolog()



print(model)
with mlflow.start_run(run_name="run_regresion"):
    mlflow.log_param("max_iter", 200)
    
    # Log del modelo como artefacto
    mlflow.sklearn.log_model(model, artifact_path="model")
    #guarda el modelo en disco
    #mlflow.sklearn.save_model(model, 'c:/users/mlruns/models')
    #guardo tambien en mlflow
    #guarda el modelo en mlflow
    mlflow.sklearn.log_model(model, artifact_path="model")
    # Registrar el modelo (crea una versión en el registry)
    result = mlflow.register_model(
        model_uri=f"runs:/{mlflow.active_run().info.run_id}/model",
        name="IrisLogReg"
    )
    
    #cargo el modelo
    #model = mlflow.sklearn.load_model('c:/users/mamen/mlruns/models')
    print(model)
print("Registered model version:", result.version)


2026/02/05 18:07:22 INFO mlflow.tracking.fluent: Experiment with name 'iris LogisticRegression' does not exist. Creating a new experiment.


LogisticRegression(max_iter=200)


2026/02/05 18:07:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\tteru\.pyenv\pyenv-win\versions\3.12.8\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)
2026/02/05 18:07:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'IrisLogReg'.
2026/02/05 18:07:30 WARNING mlflow.tracking._model_registry.fluent: Run with id 78fb9af1d1d2457198fb8d609a71a188 has no artifacts at artifact path 'model', registering model based on models:/m-0fc69807f4504d0bba731b71425170b1 instead
2026/02/05 18:07:30 INFO mlflow.store.model_regist

LogisticRegression(max_iter=200)
🏃 View run run_regresion at: http://127.0.0.1:5000/#/experiments/4/runs/78fb9af1d1d2457198fb8d609a71a188
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Registered model version: 1


model = mlflow.sklearn.load_model(f"runs:/{run_id}/tracking_path")

#Training Data
X_train, X_test, y_train, y_test = train_test_split(X, y,train_size=0.7,random_state=0)
#Linear Regression model
lr = LinearRegression()
lr.fit(X_train, y_train)

#Dataset 
eval_data = X_test 
eval_data["test_label"] = y_test
#Evaluate model with Dataset
mlflow.evaluate("runs:/run_id/model",eval_data,targets="test_label",model_type="regressor")

### Predict

In [10]:
import pandas as pd

model_uri = f"runs:/run_regresion/model"       # 'model' = artifact_path donde guardaste el modelo

model = mlflow.sklearn.load_model(model_uri)  # carga estimador sklearn nativo :contentReference[oaicite:3]{index=3}

# Ejemplo de datos (ajusta columnas/orden a tu entrenamiento)
X_new = pd.DataFrame([
    {"sepal length (cm)": 5.1, "sepal width (cm)": 3.5, "petal length (cm)": 1.4, 
     "petal width (cm)": 0.2}
])

y_pred = model.predict(X_new)
print(y_pred)

RestException: RESOURCE_DOES_NOT_EXIST: Run with id=run_regresion not found